%% [markdown]
# ESM-2 backprop / gradient-flow parity — CPU / CUDA / Trainium  (INTEGRATION mode)
The integration-critical test: run one backward through a **composite (ESM-2 backbone + a new
downstream head)** on every device and check the gradients match — proving that when you use ESM-2
embeddings as inputs to a new model, gradients flow correctly through the backbone **and** into the
new head on Trainium (so the new model can train / the backbone can be finetuned).

Pin a free core: `NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 02_backprop_parity.ipynb`.
Gradient parity is **magnitude-aware** (near-zero grads are fp noise where cosine is meaningless).

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import esm2_reference as R

[W708 23:24:29.117462092 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES)

torch 2.11.0+cpu | devices: ['cpu', 'neuron']


In [4]:
# %% [markdown]
# ## One backward per device through the composite; collect grads (backbone + head)
# %%
def run_backprop(device):
    model = R.load(device, finetune_backbone=True)   # backbone trainable -> grads flow all the way
    for p in model.parameters():
        p.requires_grad_(True)
    model.zero_grad(set_to_none=True)
    ids, mask = R.build_inputs()
    out = model(ids.to(device), mask.to(device))
    R.loss_fn(out).backward()
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    grads = {n: p.grad.detach().float().cpu() for n, p in model.named_parameters() if p.grad is not None}
    return grads

In [5]:
results = {d: run_backprop(d) for d in DEVICES}
for d in DEVICES:
    print(f"{d:7s} grad_tensors={len(results[d])}")

/home/user/miniconda3/envs/torch-neuron/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 14480.74it/s]


[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 15180.04it/s]


[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cpu     grad_tensors=101
neuron  grad_tensors=101


In [6]:
# %% [markdown]
# ## Magnitude-aware gradient comparison vs CPU (report head vs backbone separately)
# %%
def compare(ref, test):
    scale = max(g.abs().max().item() for g in ref.values()) or 1.0
    real = []
    for n, gr in ref.items():
        a, b = gr.flatten(), test[n].flatten()
        if F.cosine_similarity(a, b, dim=0).item() < 0.99 and (a - b).abs().max().item() / scale > 1e-3:
            real.append((n, (a - b).abs().max().item()))
    return scale, real

In [7]:
ref, all_ok = results["cpu"], True
for d in DEVICES:
    if d == "cpu":
        continue
    scale, real = compare(ref, results[d])
    n_head = sum(1 for n in ref if n.startswith("head."))
    n_bb = len(ref) - n_head
    print(f"\n{d} vs cpu: {len(ref)} grad tensors ({n_bb} backbone + {n_head} new-head) | global |grad|={scale:.3e}")
    print(f"  matched: {len(ref) - len(real)}/{len(ref)}   (gradient flows through backbone AND into the new head)")
    for n, ad in real:
        print(f"  REAL DIFF {n}  absdiff={ad:.3e}")
    all_ok = all_ok and not real


neuron vs cpu: 101 grad tensors (99 backbone + 2 new-head) | global |grad|=1.974e+03
  matched: 101/101   (gradient flows through backbone AND into the new head)


In [8]:
print("\nGRADIENT-FLOW PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "gradients diverged across devices (beyond near-zero fp noise)"


GRADIENT-FLOW PARITY: PASS
